In [209]:
from langgraph.graph import START, END, StateGraph
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv
load_dotenv()

True

In [210]:
model = ChatGroq(
    model="openai/gpt-oss-safeguard-20b",
    temperature=0.7,
    max_tokens=4096
)

In [211]:
class EssayBuilder(TypedDict):
    topic:str
    outline:str
    content:str
    evaluate:str
    

In [212]:
def create_outline(state:EssayBuilder)-> EssayBuilder:
    # Fetch the topic 
    topic = state['topic']  

    # call the llm gen outline
    prompt = f"Write an outline for an essay on {topic}"
    outline = str(model.invoke(prompt).content)

    # update state
    state['outline'] = outline
    return state

In [213]:
def create_content(state:EssayBuilder)-> EssayBuilder:
    # Fetch the topic 
    content = state['content']  
    outline = state['outline']

    # call the llm gen content
    prompt = f"Write an body for an essay based on {outline}"
    content = str(model.invoke(prompt).content)

    # update state
    state['content'] = content
    return state

In [214]:
def evaluate_content(state:EssayBuilder)-> EssayBuilder:
    # Fetch the topic 
    content = state['content']  
    outline = state['outline']

    # call the llm gen content
    prompt = f"Evaluate the essay score based on {outline} and {content}"
    evaluate = str(model.invoke(prompt).content)

    # update state
    state['evaluate'] = evaluate
    return state

In [215]:
graph = StateGraph(EssayBuilder)

graph.add_node("create_outline", create_outline)
graph.add_node("create_content", create_content)
graph.add_node("evaluate_content", evaluate_content)

graph.add_edge(START, "create_outline")
graph.add_edge("create_outline","create_content")
graph.add_edge("create_content", "evaluate_content")
graph.add_edge("evaluate_content", END)

workflow = graph.compile()

initial_state = EssayBuilder(topic="The impact of AI on society", outline="", content="",evaluate="")
final_state = workflow.invoke(initial_state)
# print(final_state)
print(f"Essay Topic: {final_state['topic']}")
print(f"Essay Evaluation: {final_state['evaluate']}")
# print(f"Essay Outline: {final_state['outline']}")
# print(f"Essay Content: {final_state['content']}")


Essay Topic: The impact of AI on society
Essay Evaluation: **Essay Score: 9.2 / 10**

---

## 1. Overall Impression  
The essay demonstrates a **strong command of the topic**, a well‑structured outline, and a compelling narrative that balances optimism with caution. The writer successfully turns the outline into a polished, evidence‑rich discussion that covers economic, social, ethical, health, and future‑oriented dimensions of AI. The use of concrete, up‑to‑date examples and a robust reference list adds credibility and depth.

---

## 2. Scoring Breakdown (out of 10)

| Criterion | Weight | Score | Comments |
|-----------|--------|-------|----------|
| **Structure & Organization** | 1.5 | 1.5 | The essay follows the outline flawlessly. Each section has a clear purpose and logical flow. |
| **Thesis & Argument Development** | 1.5 | 1.5 | The thesis is stated early and revisited in the conclusion. Arguments are consistently tied back to it. |
| **Depth & Breadth of Content** | 2.0 | 1.8